# DICOM to Key-Frame Extraction and Angiogram Preprocessing Pipeline

 Overview

This notebook implements the **DICOM → Key Frame → Angiogram Preprocessing** pipeline
designed for the *AI-Driven Coronary Disease Detection and Decision Support System*.

The goal is to:
- Convert angiogram DICOM files into individual frames
- Automatically select diagnostically relevant key frames
- Apply standardized angiogram preprocessing
- Prepare outputs for the blockage detection and localization module


## Step 1: Upload Angiogram (DICOM)

In [1]:
from pathlib import Path

# Path to the uploaded DICOM file
dicom_path = Path("sample_angiogram.dcm")

print(f"DICOM file selected: {dicom_path}")

DICOM file selected: sample_angiogram.dcm


The system accepts angiogram data in **DICOM format**, which is the standard
medical imaging format used in hospitals. This file may contain multiple frames
representing a complete angiography sequence.

## Step 2: Validate DICOM Format

In [2]:
import pydicom

def validate_dicom(file_path: Path) -> bool:
    """
    Validates whether the given file is a readable DICOM file.
    """
    try:
        pydicom.dcmread(file_path, stop_before_pixels=True)
        return True
    except Exception as e:
        print(f"Invalid DICOM file: {e}")
        return False


# Validate input file
is_valid = validate_dicom(dicom_path)
print("DICOM validation result:", is_valid)

Invalid DICOM file: [Errno 2] No such file or directory: 'sample_angiogram.dcm'
DICOM validation result: False


Before processing, the system validates whether the uploaded file is a valid
DICOM file. This prevents corrupted or unsupported files from entering the
pipeline.

## Step 3: Read DICOM Metadata & Frames

In [3]:
import numpy as np
import pydicom
from pathlib import Path

def read_dicom_or_simulate(file_path: Path):
    """
    Reads a DICOM file if available.
    If not, generates simulated angiogram frames for development/testing.
    """

    if file_path.exists():
        ds = pydicom.dcmread(file_path)

        metadata = {
            "Source": "DICOM",
            "Modality": ds.get("Modality", "Unknown"),
            "Rows": ds.get("Rows", None),
            "Columns": ds.get("Columns", None),
            "NumberOfFrames": ds.get("NumberOfFrames", 1)
        }

        frames = ds.pixel_array.astype(np.float32)

    else:
        # ---- SIMULATION MODE ----
        print("DICOM file not found. Using simulated angiogram frames.")

        num_frames = 20
        height, width = 512, 512

        frames = np.random.normal(
            loc=100, scale=25, size=(num_frames, height, width)
        ).astype(np.float32)

        metadata = {
            "Source": "Simulated",
            "Modality": "XA",
            "Rows": height,
            "Columns": width,
            "NumberOfFrames": num_frames
        }

    return metadata, frames


metadata, frames = read_dicom_or_simulate(dicom_path)

print("Metadata:", metadata)
print("Frames shape:", frames.shape)

DICOM file not found. Using simulated angiogram frames.
Metadata: {'Source': 'Simulated', 'Modality': 'XA', 'Rows': 512, 'Columns': 512, 'NumberOfFrames': 20}
Frames shape: (20, 512, 512)


## Step 3: Read DICOM Metadata and Frames



Once validated, the DICOM file is read to extract:
- Image metadata (useful for traceability)
- Multi-frame pixel data representing the angiogram sequence

## Step 4: Extract Individual Frames

In [4]:
def extract_frames(frame_array: np.ndarray):
    """
    Splits multi-frame DICOM pixel array into individual frames.
    """
    return [frame_array[i] for i in range(frame_array.shape[0])]


frame_list = extract_frames(frames)

print(f"Total frames extracted: {len(frame_list)}")

Total frames extracted: 20


Angiogram DICOM files contain multiple frames captured over time.
Each frame is extracted and processed individually for quality analysis.